# 03 v2 Corrected Engineered Features and Embeddings

This v2 notebook creates corrected G1-G8 feature artifacts and frozen DistilBERT embeddings, including `val_adv_30`. If `adversarial_validation/val_adv_30.csv` is missing, it is generated deterministically from `val_clean` only. Download `notebook03_outputs_v2.zip` at the end before running Notebook 03b v2 in a separate runtime.

## Dependency Check

In [ ]:
import importlib.util, subprocess, sys
required = {
    "pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn",
    "torch": "torch", "transformers": "transformers", "matplotlib": "matplotlib",
    "tqdm": "tqdm", "tabulate": "tabulate"
}
missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Missing packages installed:" if missing else "All required packages available.", missing)


## Paths and Split Configuration

In [ ]:

from pathlib import Path
import json, os, random, re, time, zipfile, shutil, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

FEATURE_TEXT_COL = "message_raw"   # for engineered G1-G8 feature extraction
EMBED_TEXT_COL = "model_text"      # for frozen DistilBERT embeddings
LABEL_COL = "label_id"

assert FEATURE_TEXT_COL == "message_raw", "Engineered G1-G8 features must use message_raw."
assert EMBED_TEXT_COL == "model_text", "DistilBERT embeddings must use model_text."
assert LABEL_COL == "label_id", "Labels must use label_id."

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATASET_ZIP = CONTENT_DIR / "06_model_ready.zip"
if DATASET_ZIP.exists():
    print(f"Found dataset ZIP, extracting: {DATASET_ZIP}")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(CONTENT_DIR)
else:
    print("No /content/06_model_ready.zip found. Looking for an existing extracted dataset.")

def find_data_dir():
    candidates = [
        CONTENT_DIR / "06_model_ready",
        CONTENT_DIR / "data" / "06_model_ready",
        CONTENT_DIR / "thesis-modeling" / "data" / "06_model_ready",
        Path.cwd() / "06_model_ready",
        Path.cwd() / "data" / "06_model_ready",
        *[p / "data" / "06_model_ready" for p in [Path.cwd(), *Path.cwd().parents]],
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError("Could not find 06_model_ready. Upload /content/06_model_ready.zip or run from the thesis project root.")

DATA_DIR = find_data_dir()
# Project root is the parent of data/06_model_ready when possible; otherwise use /content.
if DATA_DIR.parent.name == "data":
    BASE_DIR = DATA_DIR.parent.parent.resolve()
else:
    BASE_DIR = CONTENT_DIR.resolve()

ARTIFACTS_DIR = BASE_DIR / "artifacts"
RESULTS_DIR = BASE_DIR / "results"
REPORTS_DIR = BASE_DIR / "reports"
MODELS_DIR = BASE_DIR / "trained_models"
FEATURE_DIR = ARTIFACTS_DIR / "features"
EMBED_DIR = ARTIFACTS_DIR / "embeddings"
METADATA_DIR = ARTIFACTS_DIR / "metadata"
GA_DIR = ARTIFACTS_DIR / "ga_runs"
METRICS_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
FIGURE_DIR = RESULTS_DIR / "figures"
DEGRADATION_DIR = RESULTS_DIR / "degradation_tables"
for d in [FEATURE_DIR, EMBED_DIR, METADATA_DIR, GA_DIR, MODELS_DIR, METRICS_DIR, PRED_DIR, FIGURE_DIR, DEGRADATION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
SPLITS = {
    "train_clean": DATA_DIR / "clean" / "train_clean.csv",
    "val_clean": DATA_DIR / "clean" / "val_clean.csv",
    "test_clean": DATA_DIR / "clean" / "test_clean.csv",
    "val_adv_10": DATA_DIR / "adversarial_validation" / "val_adv_10.csv",
    "val_adv_20": DATA_DIR / "adversarial_validation" / "val_adv_20.csv",
    "val_adv_30": DATA_DIR / "adversarial_validation" / "val_adv_30.csv",
    "test_adv_10": DATA_DIR / "adversarial_test" / "test_adv_10.csv",
    "test_adv_20": DATA_DIR / "adversarial_test" / "test_adv_20.csv",
    "test_adv_30": DATA_DIR / "adversarial_test" / "test_adv_30.csv",
}
EVAL_SPLITS = ["test_clean", "test_adv_10", "test_adv_20", "test_adv_30"]
GA_VALIDATION_SPLITS = ["val_clean", "val_adv_10", "val_adv_20", "val_adv_30"]
G_FEATURES = [
    "G1_URL_Signals", "G2_OTP_Numeric_Density", "G3_Obfuscation",
    "G4_Urgency_Threat_Cues", "G5_Action_Directives", "G6_Financial_Terms",
    "G7_Auth_Secrets_Request", "G8_Brand_Impersonation",
]
for split, path in SPLITS.items():
    if split == "val_adv_30":
        continue
    if not path.exists():
        raise FileNotFoundError(f"Missing required split {split}: {path}")
print("DATA_DIR =", DATA_DIR)
print("BASE_DIR =", BASE_DIR)


## Create or Load val_adv_30

In [ ]:

VAL_ADV_30_PATH = DATA_DIR / "adversarial_validation" / "val_adv_30.csv"
VAL_ADV_30_PATH.parent.mkdir(parents=True, exist_ok=True)

def perturb_smishing_surface_v2(text, rng):
    s = "" if pd.isna(text) else str(text)
    leet = str.maketrans({"a":"@", "A":"@", "e":"3", "E":"3", "i":"1", "I":"1", "o":"0", "O":"0", "s":"$", "S":"$"})
    homoglyph = {
    "a": "а",  # Cyrillic a
    "e": "е",  # Cyrillic e
    "o": "о",  # Cyrillic o
    "p": "р",  # Cyrillic er
    "c": "с",  # Cyrillic es
    "x": "х",  # Cyrillic ha
    }  # Cyrillic lookalikes
    tokens = s.split()
    out = []
    for tok in tokens:
        new = tok
        r = rng.random()
        if r < 0.25 and len(tok) >= 4:
            new = tok.translate(leet)
        elif r < 0.45 and len(tok) >= 4:
            chars = [homoglyph.get(ch, ch) if rng.random() < 0.25 else ch for ch in tok]
            new = "".join(chars)
        elif r < 0.65 and len(tok) >= 5:
            cut = max(1, min(len(tok)-1, len(tok)//2))
            new = tok[:cut] + " " + tok[cut:]
        elif r < 0.80:
            new = "".join(ch.upper() if rng.random() < 0.35 else ch.lower() for ch in tok)
        out.append(new)
    noisy = "  ".join(out)
    if rng.random() < 0.70:
        noisy += rng.choice([" !!", " ...", " !", "  "])
    return noisy

if VAL_ADV_30_PATH.exists():
    print("Existing val_adv_30 found:", VAL_ADV_30_PATH)
    val_adv_30_df = pd.read_csv(VAL_ADV_30_PATH)
    generation_mode = "loaded_existing"
else:
    print("Generating val_adv_30 from val_clean only:", VAL_ADV_30_PATH)
    rng = np.random.default_rng(3030)
    val_clean_source = pd.read_csv(SPLITS["val_clean"])
    for col in [FEATURE_TEXT_COL, EMBED_TEXT_COL, LABEL_COL]:
        assert col in val_clean_source.columns, f"val_clean missing {col}"
    val_adv_30_df = val_clean_source.copy()
    smish_mask = val_adv_30_df[LABEL_COL].astype(int) == 1
    val_adv_30_df.loc[smish_mask, FEATURE_TEXT_COL] = val_adv_30_df.loc[smish_mask, FEATURE_TEXT_COL].apply(lambda x: perturb_smishing_surface_v2(x, rng))
    val_adv_30_df.loc[smish_mask, EMBED_TEXT_COL] = val_adv_30_df.loc[smish_mask, EMBED_TEXT_COL].apply(lambda x: perturb_smishing_surface_v2(x, rng))
    val_adv_30_df["artifact_purpose"] = "adversarial_validation_v2"
    val_adv_30_df["perturbation_level"] = 30
    val_adv_30_df["perturbation_techniques"] = "homoglyph, leetspeak, token_splitting, case_manipulation, punctuation_spacing_noise"
    val_adv_30_df["label_preserved"] = True
    val_adv_30_df.to_csv(VAL_ADV_30_PATH, index=False)
    generation_mode = "generated_from_val_clean_only"

SPLITS["val_adv_30"] = VAL_ADV_30_PATH
val_adv_30_copy = METADATA_DIR / "generated_val_adv_30.csv"
val_adv_30_df.to_csv(val_adv_30_copy, index=False)
val30_config = {
    "split": "val_adv_30",
    "mode": generation_mode,
    "source_split": "val_clean",
    "test_data_used": False,
    "ham_rows_unchanged": True,
    "smishing_rows_perturbed": True,
    "seed": 3030,
    "techniques": ["homoglyph substitution", "leetspeak substitution", "token splitting", "case manipulation", "punctuation/spacing noise"],
    "output_csv": str(VAL_ADV_30_PATH),
    "metadata_copy": str(val_adv_30_copy),
}
(METADATA_DIR / "val_adv_30_generation_config.json").write_text(json.dumps(val30_config, indent=2), encoding="utf-8")
(REPORTS_DIR / "03_val_adv_30_generation_summary.md").write_text(
    "# val_adv_30 Generation Summary\n\n"
    f"- Mode: {generation_mode}\n"
    "- Source split: val_clean only.\n"
    "- Test data used: no.\n"
    "- Ham rows unchanged: yes.\n"
    "- Smishing rows perturbed: yes.\n"
    "- Techniques: homoglyph substitution, leetspeak substitution, token splitting, case manipulation, punctuation/spacing noise.\n"
    f"- Rows: {len(val_adv_30_df)}\n",
    encoding="utf-8"
)
val_adv_30_df[[FEATURE_TEXT_COL, EMBED_TEXT_COL, LABEL_COL]].head()


## Corrected G1-G8 Feature Helpers

In [ ]:
URL_RE = re.compile(r"(?ix)(https?://|www\.|bit\.ly|tinyurl|t\.co|goo\.gl|ow\.ly|is\.gd|cutt\.ly|rebrand\.ly|shorturl|lnkd\.in|[a-z0-9][a-z0-9-]{1,}\.(?:com|net|org|ph|co|io|ly|me|info|biz|site|top|xyz|click|online)\b|\b(?:link|url|http|https|dot com|\.com/|/[a-z0-9]{4,})\b)")
OTP_RE = re.compile(r"(?ix)\b(?:otp|o\s*t\s*p|one[-\s]?time password|verification code|security code|auth(?:entication)? code|passcode|pin|p\.?i\.?n\.?|ref(?:erence)?(?: no| number)?|code)\b|\b\d{4,8}\b")
LEET_RE = re.compile(r"(?i)\b[a-z]*[0134578][a-z0-9]*\b")
SPACED_TOKEN_RE = re.compile(r"(?i)(?:\b[a-z]\s+){2,}[a-z]\b")
MIXED_CASE_RE = re.compile(r"\b(?=\w*[a-z])(?=\w*[A-Z])\w{4,}\b")
UNUSUAL_SYMBOL_RE = re.compile(r"[^\w\s.,!?;:'\"@#%&*()+\-=/\\]")
URGENCY_RE = re.compile(r"(?ix)\b(?:urgent|immediately|immediate|now|today|final notice|last chance|suspend(?:ed)?|block(?:ed)?|lock(?:ed)?|expire(?:d|s|ing)?|penalty|unauthorized|risk|warning|alert|limited time|avoid|deactivate|restricted|fraud|compromise(?:d)?|verify now|act now)\b")
ACTION_RE = re.compile(r"(?ix)\b(?:click|tap|open|visit|verify|confirm|update|login|log in|sign in|claim|call|reply|send|submit|activate|validate|download|install|follow|complete|provide|enter|check)\b")
FINANCIAL_RE = re.compile(r"(?ix)\b(?:bank|account|card|credit|debit|payment|refund|wallet|loan|cash|balance|transfer|transaction|remittance|prize|reward|bonus|deposit|withdraw|atm|gcash|paymaya|maya|paypal|invoice|billing|tax|pancard|pan card)\b")
AUTH_RE = re.compile(r"(?ix)\b(?:password|passcode|pin|p\.?i\.?n\.?|otp|o\s*t\s*p|cvv|cvc|verification code|security code|login credentials|credential(?:s)?|authentication|one[-\s]?time password|reset password|2fa|mfa|secret answer|recovery code)\b")
BRAND_RE = re.compile(r"(?ix)\b(?:bdo|bpi|metrobank|landbank|unionbank|security bank|china bank|rcbc|sbi|hdfc|axis bank|citibank|chase|wells fargo|bank of america|gcash|maya|paymaya|paypal|venmo|cash app|globe|smart|tm|tnt|pldt|dhl|fedex|ups|lbc|jrs|ninja van|philpost|usps|gov|sss|bir|philhealth|pag-ibig|irs|apple|google|microsoft|amazon|netflix|facebook|meta|instagram|whatsapp|telegram|twitter|x\.com|shopee|lazada|grab|foodpanda|yono|pancard)\b")

def load_split(split):
    df = pd.read_csv(SPLITS[split])
    assert FEATURE_TEXT_COL in df.columns, f"{split} is missing {FEATURE_TEXT_COL}"
    assert EMBED_TEXT_COL in df.columns, f"{split} is missing {EMBED_TEXT_COL}"
    assert LABEL_COL in df.columns, f"{split} is missing {LABEL_COL}"
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df

def assert_required_columns(df, split):
    assert "message_raw" in df.columns, f"{split} must contain message_raw"
    assert "model_text" in df.columns, f"{split} must contain model_text"
    assert "label_id" in df.columns, f"{split} must contain label_id"

def norm_count(count, denom, cap=3.0):
    return 0.0 if denom <= 0 else float(min(count / denom, cap) / cap)

def score_text(text):
    s = "" if pd.isna(text) else str(text)
    chars = max(len(s), 1)
    tokens = re.findall(r"\b\w+\b", s)
    token_count = max(len(tokens), 1)
    digit_count = sum(ch.isdigit() for ch in s)
    number_tokens = sum(1 for t in tokens if any(ch.isdigit() for ch in t))
    url_hits = len(URL_RE.findall(s)); otp_hits = len(OTP_RE.findall(s))
    leet_hits = len(LEET_RE.findall(s)); spaced_hits = len(SPACED_TOKEN_RE.findall(s))
    mixed_hits = len(MIXED_CASE_RE.findall(s)); unusual_hits = len(UNUSUAL_SYMBOL_RE.findall(s))
    repeated_punct = len(re.findall(r"[!?.,;:]{3,}", s))
    fragmented = len(re.findall(r"\b\w+(?:[-_.]\w+){2,}\b", s))
    non_ascii = sum(ord(ch) > 127 for ch in s)
    urgency_hits = len(URGENCY_RE.findall(s)); action_hits = len(ACTION_RE.findall(s))
    financial_hits = len(FINANCIAL_RE.findall(s)); auth_hits = len(AUTH_RE.findall(s))
    brand_hits = len(BRAND_RE.findall(s))
    g1 = min(1.0, 0.65 * (url_hits > 0) + norm_count(url_hits, token_count, cap=0.12))
    numeric_density = 0.55 * min(digit_count / chars, 1.0) + 0.25 * min(number_tokens / token_count, 1.0)
    g2 = min(1.0, numeric_density + 0.20 * min(otp_hits, 3) / 3)
    obf_raw = leet_hits + spaced_hits + mixed_hits + repeated_punct + fragmented + min(unusual_hits, 5) + min(non_ascii, 5)
    g3 = min(1.0, 0.35 * (obf_raw > 0) + norm_count(obf_raw, token_count, cap=0.25))
    g4 = min(1.0, 0.35 * (urgency_hits > 0) + norm_count(urgency_hits, token_count, cap=0.15))
    g5 = min(1.0, 0.35 * (action_hits > 0) + norm_count(action_hits, token_count, cap=0.15))
    g6 = min(1.0, 0.35 * (financial_hits > 0) + norm_count(financial_hits, token_count, cap=0.15))
    g7 = min(1.0, 0.45 * (auth_hits > 0) + norm_count(auth_hits, token_count, cap=0.12))
    suspicious_context = max(g1, g4, g5, g6, g7)
    g8 = min(1.0, (0.30 * (brand_hits > 0) + norm_count(brand_hits, token_count, cap=0.10)) * (0.60 + 0.40 * suspicious_context))
    return {
        "G1_URL_Signals": g1, "G2_OTP_Numeric_Density": g2, "G3_Obfuscation": g3,
        "G4_Urgency_Threat_Cues": g4, "G5_Action_Directives": g5, "G6_Financial_Terms": g6,
        "G7_Auth_Secrets_Request": g7, "G8_Brand_Impersonation": g8,
        "debug_url_hits": url_hits, "debug_digit_count": digit_count, "debug_number_token_count": number_tokens,
        "debug_otp_like_hits": otp_hits, "debug_obfuscation_hits": obf_raw, "debug_urgency_hits": urgency_hits,
        "debug_action_hits": action_hits, "debug_financial_hits": financial_hits, "debug_auth_secret_hits": auth_hits,
        "debug_brand_hits": brand_hits,
    }


## Generate Corrected Feature Artifacts and Reports

In [ ]:
summary_rows, distribution_rows, selected_columns = [], [], {}
for split in SPLITS:
    df = load_split(split)
    assert_required_columns(df, split)
    text_col = FEATURE_TEXT_COL
    selected_columns[split] = text_col
    print(f"Extracting corrected G1-G8 features for {split} using {text_col}: {len(df)} rows")
    feature_texts = df["message_raw"].fillna("").astype(str)
    scored = pd.DataFrame([score_text(x) for x in tqdm(feature_texts, desc=split)])
    out = pd.DataFrame()
    for id_col in ["final_row_id", "source_final_row_id", "adversarial_id", "original_final_row_id"]:
        if id_col in df.columns: out[id_col] = df[id_col]
    out["label_id"] = df[LABEL_COL].astype(int)
    if "normalized_label" in df.columns: out["final_label"] = df["normalized_label"]
    out["feature_text_col"] = FEATURE_TEXT_COL
    out["embedding_text_col"] = EMBED_TEXT_COL
    out["feature_text_source"] = FEATURE_TEXT_COL
    out["feature_text"] = feature_texts
    out["message_raw"] = df[FEATURE_TEXT_COL].fillna("").astype(str)
    out["model_text"] = df[EMBED_TEXT_COL].fillna("").astype(str)
    out = pd.concat([out, scored], axis=1)
    assert list(out[G_FEATURES].columns) == G_FEATURES
    assert len(out) == len(df)
    assert out[G_FEATURES].isna().sum().sum() == 0
    out.to_csv(FEATURE_DIR / f"{split}_features_g1_g8.csv", index=False)
    counts = df[LABEL_COL].value_counts().sort_index().to_dict()
    summary_rows.append({"split": split, "dataset_rows": len(df), "feature_rows": len(out), "feature_columns": len(G_FEATURES), "feature_text_source": text_col, "label_0_count": int(counts.get(0, 0)), "label_1_count": int(counts.get(1, 0)), "missing_feature_values": 0})
    desc = out[G_FEATURES].agg(["min", "max", "mean", "std"]).T.reset_index(names="feature")
    desc.insert(0, "split", split)
    distribution_rows.extend(desc.to_dict("records"))

config = {"feature_order": G_FEATURES, "feature_text_col": FEATURE_TEXT_COL, "embedding_text_col": EMBED_TEXT_COL, "selected_feature_text_columns": selected_columns, "scoring": "Deterministic regex/heuristic scores clipped to [0,1]. Labels are copied only for auditing and are not used to compute scores."}
(METADATA_DIR / "feature_extraction_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
summary = pd.DataFrame(summary_rows)
pd.DataFrame(distribution_rows).to_csv(REPORTS_DIR / "03_feature_distribution_summary.csv", index=False)
(REPORTS_DIR / "03_feature_extraction_summary.md").write_text(
    "# 03 Corrected Feature Extraction Summary\n\nCorrected engineered features are generated in the exact thesis order:\n\n"
    + "\n".join(f"{i+1}. {name}" for i, name in enumerate(G_FEATURES))
    + "\n\nEngineered feature source column: message_raw\n\nDistilBERT embedding source column: model_text\n\nFeature extraction uses `message_raw` exactly. Embedding extraction uses `model_text` exactly.\n\n"
    + "## Split Checks\n\n" + summary.to_markdown(index=False)
    + "\n\n## Leakage Routing\n\nFeature extraction is deterministic preprocessing. val_adv_30 was created from val_clean only when needed. Test data was not used in feature extraction decisions, training, GA, or threshold tuning.\n",
    encoding="utf-8"
)
summary


## Extract Frozen DistilBERT Embeddings

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 128
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
encoder.eval()
print("Embedding device:", DEVICE)

@torch.no_grad()
def embed_texts(texts):
    arrays = []
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding batches"):
        batch = texts[start:start+BATCH_SIZE]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        cls = encoder(**encoded).last_hidden_state[:, 0, :].detach().cpu().numpy().astype("float32")
        arrays.append(cls)
    return np.vstack(arrays)

rows, selected_embedding_columns = [], {}
for split in SPLITS:
    df = load_split(split)
    assert_required_columns(df, split)
    text_col = EMBED_TEXT_COL
    selected_embedding_columns[split] = text_col
    print(f"Extracting frozen DistilBERT embeddings for {split} using {text_col}: {len(df)} rows")
    embedding_texts = df["model_text"].fillna("").astype(str)
    arr = embed_texts(embedding_texts.tolist())
    assert arr.shape == (len(df), 768)
    assert not np.isnan(arr).any()
    np.save(EMBED_DIR / f"{split}_distilbert_embeddings.npy", arr)
    rows.append({"split": split, "dataset_rows": len(df), "embedding_rows": int(arr.shape[0]), "embedding_dim": int(arr.shape[1]), "embedding_text_source": text_col, "missing_values": int(np.isnan(arr).sum())})

embedding_config = {"model_name": MODEL_NAME, "feature_text_col": FEATURE_TEXT_COL, "embedding_text_col": EMBED_TEXT_COL, "embedding": "Frozen DistilBERT CLS-position representation: last_hidden_state[:, 0, :]", "max_length": MAX_LENGTH, "device": DEVICE, "expected_embedding_dim": 768, "selected_embedding_text_columns": selected_embedding_columns}
(METADATA_DIR / "embedding_extraction_config.json").write_text(json.dumps(embedding_config, indent=2), encoding="utf-8")
embedding_shape_check = pd.DataFrame(rows)
embedding_shape_check.to_csv(REPORTS_DIR / "03_embedding_shape_check.csv", index=False)
embedding_shape_check


## Create Notebook 03 Output ZIP

In [ ]:

import zipfile
from pathlib import Path
import shutil

zip_path = CONTENT_DIR / "notebook03_outputs_v2.zip"
if zip_path.exists():
    zip_path.unlink()
folders_to_zip = ["artifacts/features", "artifacts/embeddings", "artifacts/metadata", "reports"]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in folders_to_zip:
        src = BASE_DIR / rel
        if src.exists():
            for file in src.rglob("*"):
                if file.is_file():
                    zf.write(file, file.relative_to(BASE_DIR))
        else:
            print(f"Skipping missing folder: {src}")
print(f"? Notebook 03 output ZIP created: {zip_path}")
print("Download this file and upload it before running Notebook 03b if using a separate runtime.")


## Notebook 03 ZIP Checklist

In [ ]:

required_entries = ["artifacts/features", "artifacts/embeddings", "artifacts/metadata", "reports"]
with zipfile.ZipFile(zip_path, "r") as zf:
    names = set(zf.namelist())
    for entry in required_entries:
        ok = any(name.startswith(entry.rstrip("/") + "/") or name.rstrip("/") == entry.rstrip("/") for name in names)
        print(f"{entry}: {'FOUND' if ok else 'MISSING'}")
